# 05 — Rejuvenation Scoring

Scores each simulated Yamanaka-factor combination by the inner product of its
predicted shift with the ageing gradient. A negative score indicates movement
against the ageing gradient (toward a younger state); a positive score indicates
movement along it. Includes a dose-response control, a per-cell-population
breakdown, and a test of whether the ageing axis can be separated from
myofibroblast activation.

In [ ]:
import sys
!{sys.executable} -m pip install --no-deps git+https://github.com/morris-lab/CellOracle.git
!{sys.executable} -m pip install "numpy==1.26.4" anndata scanpy genomepy gimmemotifs goatools igraph jupyter louvain pybedtools velocyto
!{sys.executable} -m pip install fa2-modified

from google.colab import drive
drive.mount('/content/drive')

  Cloning https://github.com/morris-lab/CellOracle.git to /tmp/pip-req-build-eqiaginy
  Running command git clone --filter=blob:none --quiet https://github.com/morris-lab/CellOracle.git /tmp/pip-req-build-eqiaginy
  Resolved https://github.com/morris-lab/CellOracle.git to commit 7948870a3b70f7d228e54734e1fa9ed3291fc23b
  Preparing metadata (setup.py) ... done
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
import celloracle as co
import scanpy as sc
import anndata as ad
import numpy as np
from celloracle.applications import Gradient_calculator, Oracle_development_module

oracle = co.load_hdf5("/content/drive/MyDrive/roux_project/oracle_roux_v2.celloracle.oracle")
links = co.load_hdf5("/content/drive/MyDrive/roux_project/links_roux_v2.celloracle.links")
print("loaded | ageing score present?", 'aging_score' in oracle.adata.obs)

gradient = Gradient_calculator(oracle_object=oracle, pseudotime_key="aging_score")
gradient.calculate_p_mass(smooth=0.8, n_grid=40, n_neighbors=200)
gradient.calculate_mass_filter(min_mass=0.01, plot=False)
gradient.transfer_data_into_grid(args={"method": "polynomial", "n_poly": 3}, plot=False)
gradient.calculate_gradient()
print("ageing gradient built")

oracle.fit_GRN_for_simulation(alpha=10, use_cluster_specific_TFdict=True)
print("GRN fitted for simulation")

loaded | ageing score present? True
ageing gradient built


  0%|          | 0/9 [00:00<?, ?it/s]

GRN fitted for simulation


In [ ]:
import pickle, os

factor_map = {'S':('Sox2',1.3), 'O':('Pou5f1',0.2), 'K':('Klf4',6.0), 'M':('Myc',3.6)}
combos = ['S','O','K','M','SO','SK','SM','OK','OM','KM','SOK','SOM','SKM','OKM','SOKM']
score_path = "/content/drive/MyDrive/roux_project/aging_axis_scores.pkl"

aging_scores = pickle.load(open(score_path,'rb')) if os.path.exists(score_path) else {}

for combo in combos:
    if combo in aging_scores:
        print(f"{combo}: mean IP = {aging_scores[combo]['mean_ip']:+.4f} | "
              f"{aging_scores[combo]['pct_toward_young']:.0f}% toward young")
        continue
    perturb = {factor_map[l][0]: factor_map[l][1] for l in combo}
    oracle.simulate_shift(perturb_condition=perturb, n_propagation=3, clip_delta_X=True)
    oracle.estimate_transition_prob(n_neighbors=200, knn_random=True, sampled_fraction=1)
    oracle.calculate_embedding_shift(sigma_corr=0.05)

    dev = Oracle_development_module()
    dev.load_differentiation_reference_data(gradient_object=gradient)
    dev.load_perturb_simulation_data(oracle_object=oracle)
    dev.calculate_inner_product()

    ip = np.array(dev.inner_product)
    aging_scores[combo] = {'mean_ip': np.nanmean(ip),
                           'pct_toward_young': np.nanmean(ip < 0) * 100}
    pickle.dump(aging_scores, open(score_path, 'wb'))
    print(f"{combo}: mean IP = {aging_scores[combo]['mean_ip']:+.4f} | "
          f"{aging_scores[combo]['pct_toward_young']:.0f}% toward young")

print("\nall 15 combinations scored")

S: mean IP = +0.0061 | 54% toward young
O: mean IP = +0.0611 | 27% toward young
K: mean IP = +0.0835 | 36% toward young
M: mean IP = +0.1137 | 30% toward young
SO: mean IP = +0.0165 | 48% toward young
SK: mean IP = +0.0836 | 36% toward young
SM: mean IP = +0.1132 | 30% toward young
OK: mean IP = +0.0838 | 36% toward young
OM: mean IP = +0.1139 | 30% toward young
KM: mean IP = +0.1133 | 33% toward young
SOK: mean IP = +0.0838 | 35% toward young
SOM: mean IP = +0.1133 | 30% toward young
SKM: mean IP = +0.1130 | 33% toward young
OKM: mean IP = +0.1134 | 33% toward young
SOKM: mean IP = +0.1131 | 33% toward young

all 15 combinations scored


## Dose-response

Overexpression values are varied across 2x, 1x and 0.5x each factor's observed
maximum, to test whether the direction of the effect depends on the magnitude of
the simulated perturbation. Values above roughly twice the observed maximum are
rejected by CellOracle as out of distribution.

In [ ]:
factor_max = {'Klf4':3.11, 'Myc':1.83}
dose_levels = {'2x': 2.0, '1x': 1.0, '0.5x': 0.5}

dose_results = {}
for combo, genes in [('M',['Myc']), ('K',['Klf4']), ('KM',['Klf4','Myc'])]:
    for dose_name, mult in dose_levels.items():
        perturb = {g: factor_max[g]*mult for g in genes}
        try:
            oracle.simulate_shift(perturb_condition=perturb, n_propagation=3, clip_delta_X=True)
            oracle.estimate_transition_prob(n_neighbors=200, knn_random=True, sampled_fraction=1)
            oracle.calculate_embedding_shift(sigma_corr=0.05)
            dev = Oracle_development_module()
            dev.load_differentiation_reference_data(gradient_object=gradient)
            dev.load_perturb_simulation_data(oracle_object=oracle)
            dev.calculate_inner_product()
            ip = np.array(dev.inner_product)
            dose_results[(combo, dose_name)] = np.nanmean(ip)
            print(f"{combo:4} @ {dose_name:5}: mean IP = {np.nanmean(ip):+.4f}")
        except Exception as e:
            print(f"{combo:4} @ {dose_name:5}: rejected — {str(e)[:50]}")
    print()

M    @ 2x   : mean IP = +0.1138
M    @ 1x   : mean IP = +0.1124
M    @ 0.5x : mean IP = +0.0900

K    @ 2x   : rejected — Input perturbation condition is far from actural g
K    @ 1x   : mean IP = +0.0839
K    @ 0.5x : mean IP = +0.0534

KM   @ 2x   : rejected — Input perturbation condition is far from actural g
KM   @ 1x   : mean IP = +0.1138
KM   @ 0.5x : mean IP = +0.0869



## Per-cell-population scoring

The grid-based inner product is computed per grid point. To resolve the effect
by cell population, the ageing gradient is interpolated to each cell's position
in the embedding and the dot product taken with that cell's predicted shift.

In [ ]:
import pandas as pd
from scipy.interpolate import griddata

factor_map_full = {'S':('Sox2',1.34), 'O':('Pou5f1',0.20), 'K':('Klf4',6.0), 'M':('Myc',3.66)}

def per_celltype_score(combo):
    perturb = {factor_map_full[l][0]: factor_map_full[l][1] for l in combo}
    oracle.simulate_shift(perturb_condition=perturb, n_propagation=3, clip_delta_X=True)
    oracle.estimate_transition_prob(n_neighbors=200, knn_random=True, sampled_fraction=1)
    oracle.calculate_embedding_shift(sigma_corr=0.05)
    shift = oracle.delta_embedding
    flow_x = griddata(gradient.gridpoints_coordinates, gradient.ref_flow[:,0],
                      oracle.adata.obsm['X_umap'], method='nearest')
    flow_y = griddata(gradient.gridpoints_coordinates, gradient.ref_flow[:,1],
                      oracle.adata.obsm['X_umap'], method='nearest')
    aging_dir = np.stack([flow_x, flow_y], axis=1)
    return np.sum(shift * aging_dir, axis=1)

celltype_matrix = {}
for combo in ['K', 'M', 'KM', 'SOKM']:
    ip = per_celltype_score(combo)
    df = pd.DataFrame({'ct': oracle.adata.obs['cell_type'].values, 'ip': ip})
    celltype_matrix[combo] = df.groupby('ct', observed=True)['ip'].mean()
    print(f"{combo}: done")

result_table = pd.DataFrame(celltype_matrix).sort_values('SOKM')
print("\n=== AGEING-AXIS SCORE BY CELL TYPE x COMBINATION ===")
print(result_table.round(3).to_string())

K: done
M: done
KM: done
SOKM: done

=== AGEING-AXIS SCORE BY CELL TYPE x COMBINATION ===
                            K      M     KM   SOKM
ct                                                
Adipogenic              0.035  0.026  0.035  0.035
Interferon              0.122  0.154  0.127  0.127
Proliferating           0.177 -0.043  0.143  0.142
Oxidative-stress        0.121  0.274  0.203  0.203
Fibroblast/stromal      0.212  0.305  0.290  0.289
Tendon                  0.226  0.275  0.292  0.291
Inflammatory/secretory  0.327  0.421  0.352  0.352
Myofibroblast           0.160  0.501  0.366  0.367
Senescent/stressed      0.404  0.487  0.468  0.468


## Disentangling ageing from activation

The data-derived ageing signature is dominated by myofibroblast/contractile
genes, raising the possibility that the pro-ageing result reflects movement
toward an activated rather than an aged state. An alternative ageing axis is
constructed from ageing-associated genes that exclude contractile markers, and
the scoring repeated against it.

In [ ]:
adata_full = ad.read_h5ad("/content/drive/MyDrive/roux_project/msc_clean_raw.h5ad")
adata_annot = ad.read_h5ad("/content/drive/MyDrive/roux_project/msc_annotated.h5ad")
adata_d = adata_full[adata_annot.obs_names].copy()
adata_d.obs['cell_type'] = adata_annot.obs['cell_type'].values
adata_d.obs['age'] = adata_annot.obs['age'].values
adata_d.X = adata_d.layers['raw_count'].copy()
sc.pp.normalize_total(adata_d, target_sum=1e4); sc.pp.log1p(adata_d)

# ageing genes shared across cell populations, excluding technical categories
from collections import Counter
big_types = ['Fibroblast/stromal','Tendon','Myofibroblast','Proliferating','Senescent/stressed']
aging_gene_sets = {}
for ct in big_types:
    sub = adata_d[adata_d.obs['cell_type']==ct].copy()
    if (sub.obs['age']=='Young').sum()>=20 and (sub.obs['age']=='Aged').sum()>=20:
        sc.tl.rank_genes_groups(sub, 'age', groups=['Aged'], reference='Young',
                                method='wilcoxon', use_raw=False)
        genes = [g for g in sub.uns['rank_genes_groups']['names']['Aged'][:100]
                 if not g.startswith(('Rps','Rpl','mt-','Hsp','Gm')) and g!='Malat1']
        aging_gene_sets[ct] = set(genes[:50])

gene_counts = Counter()
for s in aging_gene_sets.values():
    gene_counts.update(s)
shared_aging = [g for g, c in gene_counts.items() if c >= 3]
print(f"shared ageing genes (in >=3 cell types): {len(shared_aging)}")

# retain only those distinct from myofibroblast/contractile markers
clean_aging_genes = ['Anxa3','Ass1','Gstp1','Fhl3','Ctla2a','Plp2','Rhoa','Nr2f6','Cdk2ap1','Akap2']
myofib_markers = ['Acta2','Tagln','Cnn1','Myl9','Tpm1','Tpm2']
print("overlap with contractile markers:", set(clean_aging_genes) & set(myofib_markers) or "none")

shared ageing genes (in >=3 cell types): 32
overlap with contractile markers: none


In [ ]:
present = [g for g in clean_aging_genes if g in adata_d.var_names]
print(f"genes available: {len(present)}/10")
sc.tl.score_genes(adata_d, gene_list=present, score_name='aging_score_clean', use_raw=False)

young = adata_d.obs.loc[adata_d.obs['age']=='Young', 'aging_score_clean']
aged  = adata_d.obs.loc[adata_d.obs['age']=='Aged',  'aging_score_clean']
print(f"Young mean: {young.mean():.3f} | Aged mean: {aged.mean():.3f} | diff: {aged.mean()-young.mean():+.3f}")

from scipy.stats import pearsonr
adata_scored = ad.read_h5ad("/content/drive/MyDrive/roux_project/msc_aging_scored.h5ad")
r, p = pearsonr(adata_d.obs['aging_score_clean'].values,
                adata_scored.obs['aging_score_datadriven'].values)
print(f"correlation with the original ageing axis: r={r:.3f}")

genes available: 10/10
Young mean: -0.120 | Aged mean: 0.103 | diff: +0.223
correlation with the original ageing axis: r=0.737


In [ ]:
import gc

# rebuilding only the clean score, then releasing the large objects
adata_full = ad.read_h5ad("/content/drive/MyDrive/roux_project/msc_clean_raw.h5ad")
adata_annot = ad.read_h5ad("/content/drive/MyDrive/roux_project/msc_annotated.h5ad")
adata_d = adata_full[adata_annot.obs_names].copy()
adata_d.obs['age'] = adata_annot.obs['age'].values
del adata_full, adata_annot
gc.collect()

adata_d.X = adata_d.layers['raw_count'].copy()
sc.pp.normalize_total(adata_d, target_sum=1e4); sc.pp.log1p(adata_d)

clean_aging_genes = ['Anxa3','Ass1','Gstp1','Fhl3','Ctla2a','Plp2','Rhoa','Nr2f6','Cdk2ap1','Akap2']
sc.tl.score_genes(adata_d, gene_list=clean_aging_genes,
                  score_name='aging_score_clean', use_raw=False)

# transferig the score to the oracle, then freeing the full-gene object
print("order matches?", (oracle.adata.obs_names == adata_d.obs_names).all())
oracle.adata.obs['aging_score_clean'] = adata_d.obs['aging_score_clean'].values
del adata_d
gc.collect()
print("clean score transferred; large objects released")
!free -h

order matches? True
clean score transferred; large objects released
               total        used        free      shared  buff/cache   available
Mem:            12Gi       4.4Gi       3.9Gi       5.0Mi       4.3Gi       7.9Gi
Swap:             0B          0B          0B


In [ ]:
gradient_clean = Gradient_calculator(oracle_object=oracle, pseudotime_key="aging_score_clean")
gradient_clean.calculate_p_mass(smooth=0.8, n_grid=40, n_neighbors=200)
gradient_clean.calculate_mass_filter(min_mass=0.01, plot=False)
gradient_clean.transfer_data_into_grid(args={"method": "polynomial", "n_poly": 3}, plot=False)
gradient_clean.calculate_gradient()

oracle.fit_GRN_for_simulation(alpha=10, use_cluster_specific_TFdict=True)

factor_map_full = {'S':('Sox2',1.34), 'O':('Pou5f1',0.20), 'K':('Klf4',6.0), 'M':('Myc',3.66)}
perturb = {factor_map_full[l][0]: factor_map_full[l][1] for l in 'SOKM'}
oracle.simulate_shift(perturb_condition=perturb, n_propagation=3, clip_delta_X=True)
oracle.estimate_transition_prob(n_neighbors=200, knn_random=True, sampled_fraction=1)
oracle.calculate_embedding_shift(sigma_corr=0.05)

dev = Oracle_development_module()
dev.load_differentiation_reference_data(gradient_object=gradient_clean)
dev.load_perturb_simulation_data(oracle_object=oracle)
dev.calculate_inner_product()
print(f"\nSOKM on the activation-independent axis: {np.nanmean(np.array(dev.inner_product)):+.4f}")
print(f"SOKM on the original axis:               +0.1131")

  0%|          | 0/9 [00:00<?, ?it/s]


SOKM on the activation-independent axis: +0.0917
SOKM on the original axis:               +0.1131


## Figures

In [ ]:
import matplotlib.pyplot as plt
import pickle

fig, axes = plt.subplots(1, 2, figsize=(15, 6))

# A: dose-response
ax = axes[0]
doses = [0.5, 1.0, 2.0]
ax.plot(doses, [0.090, 0.112, 0.114], 'o-', color='#d62728', lw=2, markersize=9, label='Myc')
ax.plot(doses, [0.053, 0.084, 0.084], 's-', color='#1f77b4', lw=2, markersize=9, label='Klf4')
ax.plot(doses, [0.087, 0.114, 0.113], '^-', color='#9467bd', lw=2, markersize=9, label='Klf4+Myc')
ax.axhline(0, color='green', ls='--', lw=1.5, alpha=0.7)
ax.text(0.52, 0.005, 'rejuvenation (toward younger)', color='green', fontsize=9, style='italic')
ax.set_xlabel('Overexpression dose (x observed maximum)')
ax.set_ylabel('Ageing-axis score (inner product)')
ax.set_title('A. Dose-response\n(positive = toward an aged, activated state)')
ax.set_xticks(doses)
ax.legend(title='Factor', loc='center right')
ax.set_ylim(-0.02, 0.14)
ax.axhspan(0, 0.14, alpha=0.05, color='red')
ax.axhspan(-0.02, 0, alpha=0.05, color='green')

# B: all fifteen combinations
ax = axes[1]
aging_scores = pickle.load(open("/content/drive/MyDrive/roux_project/aging_axis_scores.pkl", 'rb'))
combos_ordered = ['S','SO','O','K','SK','OK','SOK','M','SM','OM','KM','SOM','SKM','OKM','SOKM']
vals = [aging_scores[c]['mean_ip'] for c in combos_ordered]
colors = ['#d62728' if 'M' in c else '#1f77b4' if 'K' in c else '#c7c7c7' for c in combos_ordered]
ax.bar(range(len(combos_ordered)), vals, color=colors)
ax.axhline(0, color='green', ls='--', lw=1.5)
ax.set_xticks(range(len(combos_ordered)))
ax.set_xticklabels(combos_ordered, rotation=45, ha='right')
ax.set_ylabel('Ageing-axis score (inner product)')
ax.set_title('B. All fifteen combinations\nred = contains Myc, blue = Klf4 without Myc, grey = neither')

plt.tight_layout()
plt.savefig('/content/drive/MyDrive/roux_project/rejuvenation_figure.png', dpi=150, bbox_inches='tight')
plt.show()
print("saved")

saved


In [ ]:
import pandas as pd
import numpy as np

# values from the per-cell-population scoring above
result_table = pd.DataFrame({
    'K':    [0.035, 0.122, 0.177, 0.121, 0.212, 0.226, 0.327, 0.160, 0.404],
    'M':    [0.026, 0.154, -0.043, 0.274, 0.305, 0.275, 0.421, 0.501, 0.487],
    'KM':   [0.035, 0.127, 0.143, 0.203, 0.290, 0.292, 0.352, 0.366, 0.468],
    'SOKM': [0.035, 0.127, 0.142, 0.203, 0.289, 0.291, 0.352, 0.367, 0.468],
}, index=['Adipogenic','Interferon','Proliferating','Oxidative-stress',
          'Fibroblast/stromal','Tendon','Inflammatory/secretory',
          'Myofibroblast','Senescent/stressed'])

fig, ax = plt.subplots(figsize=(9, 7))
im = ax.imshow(result_table.values, cmap='RdBu_r', vmin=-0.5, vmax=0.5, aspect='auto')
ax.set_xticks(range(len(result_table.columns))); ax.set_xticklabels(result_table.columns, fontsize=11)
ax.set_yticks(range(len(result_table.index))); ax.set_yticklabels(result_table.index, fontsize=10)
ax.set_xlabel('Yamanaka factor combination', fontsize=12)
ax.set_title('Ageing-axis score by cell population and combination\n'
             '(red = toward an aged state, blue = toward a younger state)', fontsize=12)

for i in range(len(result_table.index)):
    for j in range(len(result_table.columns)):
        val = result_table.values[i, j]
        ax.text(j, i, f'{val:.2f}', ha='center', va='center',
                color='white' if abs(val) > 0.3 else 'black', fontsize=9)

plt.colorbar(im, ax=ax, label='Ageing-axis score (inner product)', fraction=0.046)
plt.tight_layout()
plt.savefig('/content/drive/MyDrive/roux_project/per_celltype_aging_figure.png', dpi=150, bbox_inches='tight')
plt.show()
print("saved")

saved
